# Agent Patterns

This notebook covers a small LangGraph agent lab:

- ReAct-style agent loop
- `ToolNode`
- checkpointing with `InMemorySaver`
- `interrupt_before` for a human-approval pause
- error recovery with retry policy

The LangGraph docs describe agents as dynamic workflows that use tools, and show `ToolNode` as the prebuilt node for tool execution. The checkpointer docs explain that checkpointing enables persistence and human-in-the-loop workflows, and the interrupts docs show how `interrupt_before` can pause a graph before a node runs. The fault-tolerance docs show retries and fallback logic. 

## Learning goals

By the end of this notebook, you should be able to:

1. Build a small ReAct-style graph.
2. Route model output into tools with `ToolNode`.
3. Persist thread state with a checkpointer.
4. Pause before tool execution for approval.
5. Retry a failing node and recover cleanly.

## 1) Install packages

In [1]:
%pip install -qU langgraph langchain langchain-core

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-tests 1.1.4 requires pytest<9.0.0,>=7.0.0, but you have pytest 9.0.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Why this notebook does not need an API key

To keep the lab runnable everywhere, the agent brain is a simple deterministic planner node.

The graph still demonstrates the same LangGraph patterns you would use with a real tool-calling LLM:
- a stateful graph
- a tool-using loop
- a checkpointer
- an approval pause
- retries for failure recovery

## 3) Import the LangGraph pieces

In [2]:
import re
from typing import Literal
from typing_extensions import TypedDict

from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy

C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4) Define two small tools

We will use:
- a calculator
- a policy lookup tool

`ToolNode` is the prebuilt LangGraph node that executes tool calls and handles state injection and error handling.

In [3]:
@tool
def calculator(expression: str) -> str:
    '''Evaluate a basic math expression such as "17 * 19".'''
    safe_globals = {"__builtins__": {}}
    safe_locals = {}
    try:
        value = eval(expression, safe_globals, safe_locals)
        return str(value)
    except Exception as e:
        return f"Calculator error: {e}"


@tool
def policy_lookup(query: str) -> str:
    '''Look up a simple internal policy answer from a tiny dictionary.'''
    policies = {
        "refund": "Refunds are allowed within 7 days with proof of purchase.",
        "returns": "Returns are allowed within 14 days if the item is unopened.",
        "approval": "Changes above the approval threshold need human review.",
    }

    text = query.lower()
    for key, value in policies.items():
        if key in text:
            return value

    return "No matching policy found."

## 5) Build a ReAct-style planner node

This node behaves like the "LLM" in a ReAct agent:

- if the question needs a tool, it emits a tool call
- if a tool result is already present, it turns that result into a final answer
- if the question is simple, it answers directly

The LangGraph agent docs show the same overall loop: an LLM node decides whether to call a tool, a tool node executes the call, and the graph loops back to the LLM until no tool call is needed.

In [4]:
def _extract_expression(text: str) -> str:
    match = re.search(r"[\d\.\+\-\*/\(\)\s]+", text)
    if match:
        expr = match.group(0).strip()
        return expr if expr else "1 + 1"
    return "1 + 1"


def _pick_policy_query(text: str) -> str:
    return text.strip()


def react_planner(state: MessagesState) -> dict:
    messages = state["messages"]
    last = messages[-1]

    if isinstance(last, ToolMessage):
        return {
            "messages": [
                AIMessage(
                    content=f"Final answer from tool result: {last.content}"
                )
            ]
        }

    if isinstance(last, HumanMessage):
        text = last.content.lower()

        if any(word in text for word in ["refund", "return", "approval", "policy"]):
            query = _pick_policy_query(last.content)
            return {
                "messages": [
                    AIMessage(
                        content="I need to check the policy tool.",
                        tool_calls=[
                            {
                                "name": "policy_lookup",
                                "args": {"query": query},
                                "id": "policy_call_1",
                                "type": "tool_call",
                            }
                        ],
                    )
                ]
            }

        if any(sym in text for sym in ["+", "-", "*", "/", "calculate", "times", "multipl", "add"]):
            expr = _extract_expression(last.content)
            return {
                "messages": [
                    AIMessage(
                        content="I need to use the calculator.",
                        tool_calls=[
                            {
                                "name": "calculator",
                                "args": {"expression": expr},
                                "id": "calc_call_1",
                                "type": "tool_call",
                            }
                        ],
                    )
                ]
            }

        if "explain" in text or "how" in text:
            return {
                "messages": [
                    AIMessage(
                        content=(
                            "A short direct answer: this graph keeps shared state, uses a tool loop, "
                            "and can persist conversation state across turns."
                        )
                    )
                ]
            }

    return {
        "messages": [
            AIMessage(
                content="I can answer directly without tools."
            )
        ]
    }

## 6) Route to tools or end

If the latest AI message contains tool calls, the graph goes to `tools`. Otherwise it stops.

This is the same conditional-edge idea shown in the LangGraph docs for agent loops.

In [5]:
def should_continue(state: MessagesState) -> Literal["tools", END]:
    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    return END

## 7) Build the graph with a checkpointer

LangGraph’s persistence docs say checkpointers keep thread-scoped state so a run can be resumed later. The checkpointer docs also show `InMemorySaver` for experimentation.

In [6]:
builder = StateGraph(MessagesState)

builder.add_node("llm_call", react_planner)
builder.add_node("tools", ToolNode([calculator, policy_lookup]))

builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, ["tools", END])
builder.add_edge("tools", "llm_call")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer, interrupt_before=["tools"])

print("Graph compiled with checkpointing and approval pause.")

Graph compiled with checkpointing and approval pause.


## 8) Run a question that needs a tool

This question should route through the calculator.

We pass a `thread_id` so the checkpointer can store and resume the thread state. The interrupts docs say `thread_id` identifies the checkpointed thread.

In [7]:
config = {
    "configurable": {
        "thread_id": "agent-demo-001"
    }
}

first = graph.invoke(
    {"messages": [HumanMessage(content="What is 17 * 19?")]},
    config=config,
)

first

{'messages': [HumanMessage(content='What is 17 * 19?', additional_kwargs={}, response_metadata={}, id='88e9689c-1069-49b0-857a-8f97d0787cd0'),
  AIMessage(content='I need to use the calculator.', additional_kwargs={}, response_metadata={}, id='8df6776f-3cf0-46ed-ae62-9f7438d95170', tool_calls=[{'name': 'calculator', 'args': {'expression': '1 + 1'}, 'id': 'calc_call_1', 'type': 'tool_call'}], invalid_tool_calls=[])]}

## 9) Approve and resume

The graph is paused before the `tools` node because we compiled it with `interrupt_before=["tools"]`. The interrupts docs say you can then resume the graph by invoking it again with `None` and the same `thread_id`. Static interrupts are mostly for debugging, but they work well as a notebook approval checkpoint.

In [8]:
resumed = graph.invoke(None, config=config)
resumed

{'messages': [HumanMessage(content='What is 17 * 19?', additional_kwargs={}, response_metadata={}, id='88e9689c-1069-49b0-857a-8f97d0787cd0'),
  AIMessage(content='I need to use the calculator.', additional_kwargs={}, response_metadata={}, id='8df6776f-3cf0-46ed-ae62-9f7438d95170', tool_calls=[{'name': 'calculator', 'args': {'expression': '1 + 1'}, 'id': 'calc_call_1', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='2', name='calculator', id='fc23723d-1854-4d05-866b-f9956ce5bcb5', tool_call_id='calc_call_1'),
  AIMessage(content='Final answer from tool result: 2', additional_kwargs={}, response_metadata={}, id='317b126a-6ac6-44c2-8d35-5940892ab818', tool_calls=[], invalid_tool_calls=[])]}

## 10) Follow-up turn that uses the same thread

Because the thread is checkpointed, a second invocation with the same `thread_id` can continue the conversation.

This is the short-term memory behavior described in the persistence docs.

In [9]:
follow_up = graph.invoke(
    {"messages": [HumanMessage(content="How did you get that result?")]},
    config=config,
)

follow_up

{'messages': [HumanMessage(content='What is 17 * 19?', additional_kwargs={}, response_metadata={}, id='88e9689c-1069-49b0-857a-8f97d0787cd0'),
  AIMessage(content='I need to use the calculator.', additional_kwargs={}, response_metadata={}, id='8df6776f-3cf0-46ed-ae62-9f7438d95170', tool_calls=[{'name': 'calculator', 'args': {'expression': '1 + 1'}, 'id': 'calc_call_1', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='2', name='calculator', id='fc23723d-1854-4d05-866b-f9956ce5bcb5', tool_call_id='calc_call_1'),
  AIMessage(content='Final answer from tool result: 2', additional_kwargs={}, response_metadata={}, id='317b126a-6ac6-44c2-8d35-5940892ab818', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How did you get that result?', additional_kwargs={}, response_metadata={}, id='86f8f2f7-6039-4fa9-9593-fcf056cf37fb'),
  AIMessage(content='A short direct answer: this graph keeps shared state, uses a tool loop, and can persist conversation state across tur

## 11) A policy question

This question should route to the policy tool instead of the calculator.

In [10]:
policy_run = graph.invoke(
    {"messages": [HumanMessage(content="What is the refund policy?")]},
    config={"configurable": {"thread_id": "agent-demo-002"}},
)

policy_run

{'messages': [HumanMessage(content='What is the refund policy?', additional_kwargs={}, response_metadata={}, id='f3acbce2-b47d-47d6-ba4b-f4fb8e495357'),
  AIMessage(content='I need to check the policy tool.', additional_kwargs={}, response_metadata={}, id='96c757c4-9201-4348-ac65-47e8da6522a0', tool_calls=[{'name': 'policy_lookup', 'args': {'query': 'What is the refund policy?'}, 'id': 'policy_call_1', 'type': 'tool_call'}], invalid_tool_calls=[])]}

## 12) Error recovery with retries

The fault-tolerance docs show `RetryPolicy(max_attempts=3)` and explain that LangGraph can inspect retry state through `runtime.execution_info.node_attempt`.

In [11]:
class RecoveryState(TypedDict):
    result: str


def flaky_node(state: RecoveryState, runtime: Runtime) -> RecoveryState:
    if runtime.execution_info.node_attempt == 1:
        raise ConnectionError("Temporary failure from a downstream service.")
    return {"result": f"Recovered on attempt {runtime.execution_info.node_attempt}"}


recovery_builder = StateGraph(RecoveryState)
recovery_builder.add_node(
    "flaky_node",
    flaky_node,
    retry_policy=RetryPolicy(max_attempts=3),
)
recovery_builder.add_edge(START, "flaky_node")
recovery_builder.add_edge("flaky_node", END)

recovery_graph = recovery_builder.compile()

recovery_graph.invoke({"result": ""})

{'result': 'Recovered on attempt 2'}

## 13) What to remember

- `ToolNode` is the standard node for tool execution.
- `InMemorySaver` gives you thread-scoped checkpointing for a notebook lab.
- `interrupt_before` gives you a human-approval pause before a node runs.
- `RetryPolicy` handles transient failures cleanly.
- A ReAct-style loop is just: think → tool call → observe → answer.

## Key takeaways

- Tool use in LangGraph is modeled as a graph loop, not a black box.
- Checkpointing makes the conversation resumable.
- Interrupts let you pause before a sensitive action.
- Retry policies keep a node from failing permanently on temporary errors. 

## References

- ToolNode: https://docs.langchain.com/oss/python/langgraph/workflows-agents#toolnode
- Checkpointers: https://docs.langchain.com/oss/python/langgraph/checkpointers
- Interrupts: https://docs.langchain.com/oss/python/langgraph/interrupts
- Fault tolerance: https://docs.langchain.com/oss/python/langgraph/fault-tolerance